# 1. Autograd in PyTorch

## 1.1 What is Autograd?

PyTorch's automatic differentiation system. It tracks every operation done with tensors and computes gradients automatically.


## `requires_grad=True`

Tells PyTorch to track that tensor. Every **weight** needs this.

```python
w = torch.randn(3, 1, requires_grad=True)  # ← tracked
X = torch.randn(8, 3)                      # ← ignored
```


## 1.2 The Computational Graph

Every operation during the forward pass becomes a **node** in the graph.

```python
h      = X @ w1
h_relu = h.relu()
y_pred = h_relu @ w2
loss   = ((y_pred - y) ** 2).mean()
```

```
X, w1 ──→ [MatMul] ──→ [ReLU] ──→ [MatMul] ──→ [MSE] ──→ loss
                                        ↑
                                       w2
```

Each node stores:
- The operation performed
- A reference to the previous node (`next_functions`)
- How to compute the local gradient (`grad_fn`)

```python
print(loss.grad_fn)                 # MeanBackward
print(loss.grad_fn.next_functions)  # PowBackward
```


## 1.3 `loss.backward()`

Traverses the graph backwards and applies the **chain rule** at each node, accumulating the gradient in `.grad` of every weight.

```python
loss.backward()

print(w1.grad)  # ∂L/∂w1
print(w2.grad)  # ∂L/∂w2
```

Internally:

```
loss → node4 → node3 → node2 → node1
  grad=1    × local_grad × local_grad × local_grad = ∂L/∂w1
```

The loss is the **end point of the graph**. It depends on the output, which depends on the activations, which depend on the weights — everything connected. Calling `.backward()` unwinds that entire history.

In [12]:
import torch
import torch.nn as nn

model = nn.Linear(3, 1)
criterion = nn.MSELoss()

X = torch.randn(8, 3)
y = torch.randn(8, 1)

y_pred = model(X)
loss = criterion(y_pred, y)
loss.backward()

for name, param in model.named_parameters():
    print(f"{name}")
    print(f"  weights:      {param.data}")
    print(f"  gradient: {param.grad}")


weight
  weights:      tensor([[-0.2030, -0.0485, -0.3182]])
  gradient: tensor([[-1.8348, -0.8932, -0.7700]])
bias
  weights:      tensor([-0.5528])
  gradient: tensor([-2.4752])


# 2. Optimizers (`tprch.optim.Name(model.parameters(), lr)`)

## 2.1 What is an Optimizer?

The optimizer has two responsibilities:

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
# - holds a reference to all weights
# - holds the hyperparameters (lr, weight_decay, betas...)
```

```python
optimizer.step()
# - reads param.grad from each weight
# - applies the update formula
# - updates param.data
```


## 2.2 `model.parameters()`

The optimizer needs to know which tensors to update. `model.parameters()` returns all weights with `requires_grad=True`.

```python
# Internally, the optimizer stores this:
params = list(model.parameters())
# [w1, w2, b1, b2...]

# On step(), it iterates over them:
for param in params:
    grad = param.grad   # ← gradient computed by backward()
    param.data -= lr * grad
```

The tensor carries everything the optimizer needs:

```python
param.data  # ← the weight value
param.grad  # ← the gradient computed by backward()
```

## 2.3 Learning Rate

Controls the **size of the step** taken during each update.

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$

- `lr` too high → overshooting, diverges
- `lr` too low → converges too slowly

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
```

## 2.4 `.step()`

Applies the update formula for every weight. The formula changes per optimizer:

| Optimizer | What `.step()` computes | Best for |
|---|---|---|
| **SGD** | `w = w - lr * grad` | baseline |
| **Momentum** | `w = w - lr * (moving average of grads)` | plateaus, local minima |
| **Adagrad** | `w = w - lr * grad / sqrt(sum of past grad²)` | sparse features |
| **RMSprop** | `w = w - lr * grad / sqrt(moving avg of grad²)` | RNNs, non-stationary |
| **Adam** | `w = w - lr * (moving avg of grad) / sqrt(moving avg of grad²)` | general purpose |
| **AdamW** | same as Adam + decoupled weight decay `- lr * λw` | transformers, LLMs |

```python
# SGD
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# SGD + Momentum
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adagrad
optimizer = torch.optim.Adagrad(model.parameters(), lr=1e-2)

# RMSprop
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-3)

# Adam
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# AdamW (recommended)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
```

## 2.5 `zero_grad()`

PyTorch **accumulates** gradients by default. Without zeroing, gradients from previous batches stack up.

```python
# Without zero_grad
loss.backward()  # w1.grad = 0.5
loss.backward()  # w1.grad = 0.5 + new_grad ← wrong!

# With zero_grad
optimizer.zero_grad()
loss.backward()  # w1.grad = new_grad ← correct
```


In [22]:
import torch
import torch.nn as nn

# Model
model  = nn.Sequential(
    nn.Linear(3, 10),
    nn.Linear(10, 1))
# Loss
criterion = nn.MSELoss()
# Optimizer Adam
optim = torch.optim.Adam(model.parameters(), lr = 0.001) 


# Visualize the porams and gradients before
print(f'=' * 50)

for name, param in model.named_parameters():
    print(f'Name: {name}, Data: {param.data}, Gradient: {param.grad}')
    print(f'\n')

# Data
X = torch.randn(3, 3)
y = torch.randn(3, 1)

# Zering the gradient before the calculos in this iteration
optim.zero_grad()

# Get the predict and new gradients
y_pred = model(X)
loss = criterion(y_pred, y)
loss.backward()

# Update the weights with the new gradient
optim.step()

# Visualize the porams and gradients after
print(f'=' * 50)
for name, param in model.named_parameters():
    print(f'Name: {name}, Data: {param.data}, Gradient: {param.grad}')
    print(f'\n')


Name: 0.weight, Data: tensor([[ 0.2962, -0.1479,  0.0742],
        [-0.5566,  0.1495, -0.1619],
        [-0.4554,  0.3542,  0.0975],
        [ 0.5055,  0.3446,  0.0345],
        [ 0.0957, -0.5421, -0.1970],
        [-0.5574, -0.2708, -0.0263],
        [ 0.5323, -0.2967, -0.5003],
        [ 0.1418,  0.2207,  0.2866],
        [-0.3382, -0.1627,  0.1553],
        [-0.1666, -0.3432,  0.2737]]), Gradient: None


Name: 0.bias, Data: tensor([ 0.3935, -0.3053,  0.0515, -0.2397, -0.1799, -0.5327,  0.0849,  0.3264,
         0.1520,  0.5250]), Gradient: None


Name: 1.weight, Data: tensor([[-0.1528,  0.1989, -0.1983, -0.2441,  0.0502,  0.2930,  0.1203, -0.1419,
         -0.2080, -0.2234]]), Gradient: None


Name: 1.bias, Data: tensor([-0.2915]), Gradient: None


Name: 0.weight, Data: tensor([[ 0.2952, -0.1489,  0.0752],
        [-0.5556,  0.1505, -0.1629],
        [-0.4564,  0.3532,  0.0985],
        [ 0.5045,  0.3436,  0.0355],
        [ 0.0967, -0.5411, -0.1980],
        [-0.5564, -0.2698, -0.0

# 3. LR Scheduler in PyTorch

## 3.1 What is it?

The scheduler automatically adjusts the learning rate of the optimizer over training. It holds a reference to the optimizer and `scheduler.step()` simply does:

```python
optimizer.param_groups[0]['lr'] = new_lr
```


## 3.2 The Dependency Chain

```
scheduler ──→ optimizer ──→ weights
    ↑               ↑
 changes lr     uses lr on step()
```

## 3.3 Main Schedulers (`torch.optim.lr_scheduler.name(optim, others)`)

```python
# Decays lr by gamma every step_size epochs
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=10, gamma=0.1
)

# Decays smoothly following a cosine curve
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs
)

# Starts small, rises to peak, then decays
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, steps_per_epoch=len(dataloader), epochs=epochs
)

# Only decays when the metric stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.1
)
```

## 3.4 Training Loop

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    for X, y in dataloader:
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()

    scheduler.step()  # ← updates lr after each epoch

    print(optimizer.param_groups[0]['lr'])  # ← current lr
```


# 4. Gradient Clipping

## 4.1 What is it?
 
Limits the **total magnitude** of gradients before the optimizer step. Prevents exploding gradients from destabilizing training.
 
In deep networks or RNNs, gradients can explode during backward:
 
```python
w.grad = 50000  # ← huge step, weights go insane
```
 
It does not cut each gradient individually. It looks at the **total norm** of all gradients together and scales proportionally. The relative directions are preserved, only the magnitude shrinks.
 
## 4.2 How it Works
 
```python
# total norm of all gradients
total_norm = sqrt(sum(grad² for all params))
 
# if it exceeds max_norm, scale everything down
if total_norm > max_norm:
    scale = max_norm / total_norm
    for param in model.parameters():
        param.grad *= scale
```
 
Concrete example with `max_norm=1.0`:
 
```
gradients:        w1.grad = 3.0,  w2.grad = 4.0
total norm:       sqrt(3² + 4²) = 5.0
 
scale = 1.0 / 5.0 = 0.2
 
w1.grad = 3.0 × 0.2 = 0.6
w2.grad = 4.0 × 0.2 = 0.8
 
total norm now = sqrt(0.6² + 0.8²) = 1.0  ✅
```
 
If the total norm is already below `max_norm`, nothing happens and the gradients are left unchanged.
 
## 4.3 Usage (`nn.utils.clip_grad_norm_`)
 
Goes **between** `backward()` and `optimizer.step()`:
 
```python
optimizer.zero_grad()
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ← here
optimizer.step()
scheduler.step()
```
 
`max_norm=1.0` is the standard value used in most cases, including Transformers and LLMs.
 

# 5. Gradient Accumulation